# NYC Climate Budgeting: Two CRISP-DM Iterations
### Business Analytics Strategy and Implementation (NBS-7170A), Assignment 3

| Item | Detail |
|---|---|
| Dataset | NYC Climate Budgeting Report: Climate Alignment Assessment and Capital Climate Investments (NYC Open Data, c99a-c5ux) |
| Business goal | Reach net-zero greenhouse gas emissions citywide by 2050 |
| Primary question | Is planned capital spending moving toward achieving net-zero by 2050? |
| Method | CRISP-DM, with GenAI help and a human checkpoint at each step (Direct, Verify, Decide, Own) |
| This notebook | A readable record of both iterations. The code and outputs are in the working notebook (assignment_3). |

## How to read this notebook

CRISP-DM is iterative. Findings in one phase can send the project back to an earlier phase. This project has gone round once so far.

| Iteration | Phase | What happened | Part |
|---|---|---|---|
| 1 | Business Understanding | Set one goal, one primary question, four questions and four charts (10 November) | Part 1 |
| 1 | Data Understanding | Downloaded and described the data, then tested each question on it (13 to 17 November) | Part 2 |
| Loop back | Evaluation of the findings | Question 3 could only be answered partly, so it was replaced (17 November) | Part 3 |
| 2 | Business Understanding | Revised Question 3 and updated the resources and risks | Part 4 |
| 2 | Data Understanding | Tested the new Question 3 on the data | Part 5 |
| Now | Where things stand | Status of each question, open decisions and next steps | Part 6 |
| Record | Decision log | Every decision, with the phase and iteration it belongs to | Part 7 |

---
# Part 1. Iteration 1: Business Understanding (starting version)

### 1.1 Objective and success criteria

| Item | Detail |
|---|---|
| Business objective | To improve the health and quality of life of New Yorkers by reducing emissions, in line with reaching net-zero emissions by 2050. |
| Analytics success | The objective is addressed by four questions, each answered by one dashboard chart that can be reproduced by SQL and Matplotlib. |
| Business success | OMB can see whether the share of planned capital spending rated aligned with Greenhouse Gas (GHG) Mitigation is rising or falling across the three plan editions. |

### 1.2 Resources, constraints and risks (iteration 1)

| Type | Detail |
|---|---|
| Resource: data | OMB's Climate Budgeting dataset, one table of 259K rows and 19 columns, covering three plan editions (FY2025, FY2026 and FY2027 Executive Budgets). |
| Resource: columns | Greenhouse Gas (GHG) Mitigation, Greenhouse Gas (GHG) Mitigation Tracking Category, Financial Plan, Fiscal Year, Fiscal Year Amount, Asset Category, Budget Line. |
| Resource: other | Data dictionary and OMB's Climate Budgeting report; MySQL, Python, JupyterLab, Matplotlib, Pandas and Claude; instructor guidance and time until 17 December 2026. |
| Constraint 1 | The data shows planned spending and ratings, not actual emissions, so it cannot show whether net-zero will be met. |
| Constraint 2 | The dashboard has four primary charts at most and must answer the primary question. |
| Constraint 3 | The report is limited to 1,250 words. |
| Risk 1 | Pending ratings may drive any trend. |
| Risk 2 | The three plan editions overlap, so they must not be pooled or added together. |
| Risk 3 | Labels may be inconsistent. |
| Risk 4 | "Aligned" needs a written definition based on OMB's ratings. |
| Risk 5 | Claude output may be wrong, so each result must be verified. |

### 1.3 The four questions (iteration 1)

**Primary question:** Is planned capital spending moving toward achieving net-zero by 2050?

| Question | Wording | Columns | Chart | How it answers the primary goal |
|---|---|---|---|---|
| Question 1 | How is the share of planned capital spending rated aligned with greenhouse gas mitigation changing across the three plan editions? | Greenhouse Gas (GHG) Mitigation, Financial Plan, Fiscal Year Amount | Stacked bar by plan edition | Shows whether a growing share of planned spending supports net-zero from one plan edition to the next. |
| Question 2 | How does aligned spending change across fiscal years within each plan edition? | Greenhouse Gas (GHG) Mitigation, Fiscal Year, Financial Plan, Fiscal Year Amount | Line chart, one line per plan edition | Shows whether aligned funding is rising or falling over the years leading toward 2050. |
| **Question 3 (original)** | **Which types of emissions-reducing work receive the most aligned funding?** | Greenhouse Gas (GHG) Mitigation Tracking Category, Greenhouse Gas (GHG) Mitigation, Fiscal Year Amount | Bar chart by tracking category | Shows what is driving the aligned spending, such as solar, bike lanes or building efficiency. |
| Question 4 | Where is the not-aligned and pending spending concentrated? | Asset Category, Budget Line, Greenhouse Gas (GHG) Mitigation, Fiscal Year Amount | Bar chart by asset category or budget line | Shows where the gap to net-zero lies, so decision-makers know where to act. |

### 1.4 Project plan

| Dates | Stage |
|---|---|
| Deadline | 17 December 2026 (submit 16 December) |
| 10 to 15 November | Finish Business Understanding and start Data Understanding |
| 16 to 22 November | Data Preparation and database design (ERM, schema) |
| 23 to 29 November | Load MySQL, write queries, build the four charts |
| 30 November to 6 December | Evaluation, deployment, and report draft |
| 7 to 16 December | Polish, references, final checks, submit |

---
# Part 2. Iteration 1: Data Understanding

### 2.1 Data collected

| Item | Detail |
|---|---|
| Source | NYC Open Data Climate Budgeting Report Dataset (c99a-c5ux) |
| File | nyc_climate_budgeting.csv, downloaded on 13 November 2026 |
| Data dictionary | Reviewed on 13 November 2026 to understand column definitions, types and terms |
| Size | 259,491 rows and 22 columns (19 on the website, plus 4 portal system columns, minus the empty `remarks` column) |
| Units | `fiscal_year_amount` is in thousands of US dollars (data dictionary) |
| Plan editions | 3: FY2025, FY2026 and FY2027 Executive Budgets. They overlap in the years they cover, so they are compared separately and never added together. |

### 2.2 Scorecard: can the data answer each question?

| Question | Can the data answer it? | Main finding | Detail |
|---|---|---|---|
| Question 1: aligned share by plan edition | Yes, with caveats | Aligned share is 12.3%, 11.5% and 25.3% in the FY2025, FY2026 and FY2027 plans | 2.3.1 |
| Question 2: aligned spending by fiscal year | Yes | Aligned spending falls within each plan, but for the shared years 2026 to 2029 the share rises with each newer plan | 2.3.2 |
| Question 3 (original): types of work by tracking category | **Partly** | The category is empty for FY2025 and covers 45.1% (FY2026) and 89.1% (FY2027) of aligned money; most money sits in broad groups | 2.3.4 |
| Question 4: where not-aligned and pending money sits | Yes | Buildings & Facilities and Vehicles hold most of the not-aligned money; one housing budget line holds 56.7% of FY2027 pending money | 2.3.5 |

### 2.3 Describe the data

**2.2.2 Shape:** 259,491 rows and 22 columns (the website lists 19 and excludes the 3 portal system columns below, plus `:version`).

| # | Column | Type (pandas) | Non-null | Meaning (data dictionary) |
|---|---|---|---|---|
| 0 | :id | str | 259,491 | Row identifier added by the portal (not in the dictionary) |
| 1 | :version | str | 259,491 | Row version added by the portal (not in the dictionary) |
| 2 | :created_at | datetime64[us, UTC] | 259,491 | When the row was created (added by the portal) |
| 3 | :updated_at | datetime64[us, UTC] | 259,491 | When the row was last updated (added by the portal) |
| 4 | published_date | str | 259,491 | Date the data was published |
| 5 | project_id | str | 259,491 | Capital Project Identification Number |
| 6 | project_description | str | 259,331 | Description of the capital project |
| 7 | budget_line | str | 259,326 | Identifier of a Capital Unit of Appropriation |
| 8 | asset_category | str | 251,591 | Type of capital project |
| 9 | ghg_mitigation | str | 259,491 | Rating for alignment with greenhouse gas mitigation |
| 10 | flood_resiliency | str | 259,491 | Rating for alignment with flood resiliency |
| 11 | heat_resiliency | str | 259,491 | Rating for alignment with heat resiliency |
| 12 | financial_plan | str | 259,491 | The associated Financial Plan (e.g. Fiscal Year 2025 Executive Budget) |
| 13 | first_fiscal_year | int64 | 259,491 | Fiscal year the record refers to |
| 14 | fiscal_year_amount | int64 | 259,491 | Funding allocated in the given fiscal year (in thousands) |
| 15 | budget_line_title | str | 259,012 | Brief description of what can be charged to the budget line |
| 16 | environmental_and_social_benefits | str | 99,694 | Benefits beyond the emissions and resiliency ratings |
| 17 | heat_vulnerability_index | str | 96,908 | Whether the project is in a high-heat area |
| 18 | flood_vulnerability_index | str | 96,897 | Whether the project is in a high-flood zone |
| 19 | ghg_mitigation_tracking_category | str | 20,919 | Climate priority tracking category for emissions work |
| 20 | flood_resiliency_tracking_category | str | 8,665 | Climate priority tracking category for flood work |
| 21 | heat_resiliency_tracking_category | str | 15,397 | Climate priority tracking category for heat work |

**Observations**

| # | Observation |
|---|---|
| 1 | Four columns are portal system columns and are not part of the dictionary. |
| 2 | `remarks` (listed on the website) is not in the file. The portal shows it as empty, so the API probably left it out. |
| 3 | `published_date` is read as text, not as a date. |
| 4 | `fiscal_year_amount` is read as an integer, although the portal publishes it as text. Amounts are in thousands. |
| 5 | `heat_vulnerability_index` and `flood_vulnerability_index` are text, not booleans. |
| 6 | Columns with fewer than 259,491 non-null values contain blanks. The tracking categories, the vulnerability flags and the benefits column are mostly blank. |

**2.2.3 Units**

| Item | Detail |
|---|---|
| Column | `fiscal_year_amount` |
| Unit | Thousands of US dollars. A value of 500 means $500 thousand. |
| Source | The data dictionary for the Climate Alignment Assessment and Capital Climate Investments dataset. |
| Date checked | 14 November 2026 |
| Note on currency | The dictionary says "thousands". The currency is taken as US dollars because the dataset covers New York City's capital budget. |
| Note on type | The portal publishes the amount as text, and pandas read it as an integer (`int64`). |

### 2.4 Question 1 results

#### 2.3.1 Results

| Item | Detail |
|---|---|
| Columns used | `ghg_mitigation`, `financial_plan`, `fiscal_year_amount` (thousands of US dollars) |
| Assumption | *Aligned* means both **Aligned** and **Aligned Component**. Since *Aligned Component* projects have emissions reduction as a secondary purpose, this assumption makes the aligned share larger than a stricter definition would. |
| Evidence | `pd.crosstab(...)`, `df.pivot_table(...)`, and the summary table below. |

#### Result

| Plan | Total | Aligned | Aligned share | Pending share |
|-------|-------------:|-------------:|--------------:|--------------:|
| FY2025 | 97,660,584 | 12,055,566 | 12.3% | 27.6% |
| FY2026 | 196,679,481 | 22,659,407 | 11.5% | 19.8% |
| FY2027 | 117,135,827 | 29,644,972 | 25.3% | 16.9% |

#### Observations

| Topic | Observation |
|---|---|
| FY2025 total | The FY2025 total (about **$97.7 billion**) matches the figure in OMB's FY2025 Climate Budgeting Report. |
| Rows versus money | Row counts and money give different pictures (aligned rows: **6.4%**, **13.3%**, and **15.0%**), so Question 1 uses funding amounts rather than project counts. |
| Plan totals | Plan totals differ because the plans cover different fiscal years (FY2026 runs to 2035). The shares are comparable, but the totals are not. |
| Rating basis | The rating basis differs between plans. **Not Rated** and **Special Cases** appear only in FY2025, while **Special Projects** appears only in FY2026 and FY2027. In FY2025, only projects over **$50,000** were evaluated. |
| Pending ratings | **Pending Rating** falls from **27.6%** to **16.9%**, so part of any rise in the aligned share may reflect ratings being completed. |
| Negative amounts | A total of **2,108 rows** have negative amounts, mostly offsets in the special categories. The reported totals are net of these values. Excluding them would give aligned shares of **11.6%**, **11.0%**, and **24.0%**, respectively, so the overall pattern remains unchanged. |

#### Decision

| Decision | Detail |
|---|---|
| Negative amounts | Keep the negative amounts and report totals on a net basis. |

#### Can the data answer Question 1?

| Answer | Detail |
|---|---|
| **Yes, with caveats.** | The aligned share can be compared across the three plans, but the plans are not rated on the same basis, and the pending and special categories remain substantial. These factors limit the strength of direct comparisons across plans. |

### 2.5 Question 2 results

#### 2.3.2 Results

| Item | Detail |
|---|---|
| Columns used | `ghg_mitigation`, `first_fiscal_year`, `financial_plan`, `fiscal_year_amount` (thousands of US dollars) |
| Assumption | Aligned means both Aligned and Aligned Component. |
| Evidence | The three pivot tables and the corrected line chart (years a plan does not cover are left blank, amounts in billions of US dollars). |
| Consistency check | The yearly aligned amounts add up to the Question 1 totals for each plan. |

**Observations**

| Topic | Observation |
|---|---|
| Within each plan | The aligned amount and share fall in the later years. This probably reflects how plans are built, with less project detail in later years. This is an interpretation, not stated in the data. |
| Across editions | For fiscal years 2026 to 2029, which appear in all three plans, the aligned share rises with each newer edition (for example, 2026: 10.7%, 18.6%, 26.1%). |
| Years not covered | Years a plan does not cover are empty, not zero, and are left out of the chart. |
| Pending ratings | The rise across editions may partly reflect fewer pending ratings. |

**Can the data answer Question 2?**

| Answer | Detail |
|---|---|
| Yes. | The trend within each plan can be shown, but the comparison across editions for the shared years is the clearer message. |

### 2.6 Question 3 (original) check

#### 2.3.4 Original Question 3 check (replaced): Types of emissions-reducing work receiving the most aligned funding

| Item | Detail |
|---|---|
| Columns used | `ghg_mitigation_tracking_category`, `ghg_mitigation`, `financial_plan`, `fiscal_year_amount` (thousands of US dollars) |
| Assumption | Aligned means both Aligned and Aligned Component. |
| Evidence | The coverage tables, the label list, the aligned amount by category and plan, and the crosstab of rating by plan for rows with a category. |
| Consistency check | Aligned money in the coverage table matches the Question 1 totals for each plan. Categorised rows by rating add up to 20,919, the non-blank count of the column. |

**Observations**

| Topic | Observation |
|---|---|
| Coverage | The tracking category is empty for all FY2025 aligned rows. It covers 45.1% of aligned money in FY2026 and 89.1% in FY2027. |
| Labels | FY2026 uses group prefixes (for example, "Energy - Solar Energy") and FY2027 does not ("Solar Energy"). Removing the prefix merges them. |
| Broad groups | In FY2027, "Net-Zero Compatible (non-DEM)" holds 73.9% of categorised aligned money. In FY2026, the two "Buildings & Facilities" labels hold 78.2%. Specific types of work, such as bike lanes and solar, are small by comparison. |
| Categories without aligned money | "Fleet Fuel Efficiency" and "Other Sustainability (non-DEM)" have rows only under other ratings. In all, 6,076 of the 20,919 categorised rows are not aligned (mostly "Not Aligned (Short-Term Benefit)"), so the chart must filter on the rating. |
| Aligned definition | In FY2027, 6,125 of the 8,375 aligned categorised rows are Aligned Component, so the chart depends on counting Aligned Component as aligned. In FY2026, 4,939 of 6,468 are Aligned. |

**Decisions**

| Topic | Decision |
|---|---|
| Merge labels | Remove the group prefix in Data Preparation. Do not add amounts across plans. |
| Chart scope | To be decided. Recommended: chart FY2027 only (89.1% coverage), state the coverage on the chart and show the broad groups as their own bars. |

**Can the data answer Question 3?**

| Answer | Detail |
|---|---|
| Partly. | It can for FY2026 and FY2027 with stated coverage, but not for FY2025. The broad groups limit what the chart can say about specific types of work. TO BE DECIDED... |

### 2.7 Question 4 check

#### 2.3.5 Question 4 check: Where not-aligned and pending spending is concentrated

| Item | Detail |
|---|---|
| Columns used | `asset_category`, `budget_line`, `budget_line_title`, `ghg_mitigation`, `financial_plan`, `fiscal_year_amount` (thousands of US dollars) |
| Assumptions | Not aligned means Not Aligned and Not Aligned (Short-Term Benefit). Pending means Pending Rating. |
| Evidence | The totals table, the asset category tables (amount, share of the plan's total for that rating, share of the category's own total), the top ten budget lines and the titles of the three standout budget lines. |
| Consistency check | Not-aligned and pending shares match Question 1 (3.5%, 3.6%, 3.9% and 27.6%, 19.8%, 16.9%). Category amounts add up to the plan totals. |

**Observations**

| Topic | Observation |
|---|---|
| Not aligned | Buildings & Facilities holds 72.7%, 45.6% and 50.7% of each plan's not-aligned money, and Vehicles holds 11.0%, 39.6% and 38.3%. |
| Vehicles | The not-aligned share of Vehicles spending rises from 18.5% to 55.5% to 80.2%, while its pending share falls from 66.3% to 39.5% to 10.3%. This may reflect ratings being completed (interpretation, not stated in the data). |
| Pending | Buildings & Facilities holds 78.5%, 76.4% and 84.7% of each plan's pending money. Budget line C 0114 alone holds 56.7% of pending money in the FY2027 plan. |
| Budget line C 0114 | Its title is "Acquisition, constr., etc. supplementary hous. prog. and support facil". The data does not say why it is pending. |
| Budget line S 0129 | Its title is "Collection trucks and equipment". It holds 24.6% of the not-aligned money in the FY2027 plan, which fits the Vehicles pattern. The reason for the rating is not stated in the data. |
| Budget line E DUMMY | Its title is "SCA FY 2025-2029 Capital Plan Feb 2025". It spans four asset categories and looks like a placeholder line, not a single project (interpretation). |
| Labels | "IT & Technology" and "It & Technology" are the same category with different capital letters (the second appears only in FY2025). Merge in Data Preparation. |
| Blank categories | The 7,900 rows with no asset category are Not Rated, so they are not part of the not-aligned or pending results. |
| Grouping | Asset category has 14 to 15 values per plan, and budget line has about 1,550 to 1,600, so asset category gives the cleaner chart. |

**Decisions**

| Topic | Decision |
|---|---|
| Grouping for the chart | Asset category. |
| Chart view | To be decided. Recommended: the FY2027 plan, with each category's share of the plan's not-aligned and pending money, since amounts in Buildings & Facilities would dwarf the rest. |
| Budget lines and asset categories | Each of the three budget lines checked spans more than one asset category, so asset category is a property of the amount row, not of the budget line. Keep this in mind for the database design. |

**Can the data answer Question 4?**

| Answer | Detail |
|---|---|
| Yes. | For all three plans, using asset category. Budget line adds detail but is too fine for a dashboard chart. |

---
# Part 3. The loop back: why Question 3 was replaced

| Item | Detail |
|---|---|
| Trigger | The Data Understanding check of the original Question 3 (2.3.4) |
| Problem 1 | The tracking category is empty for every aligned FY2025 row, so FY2025 cannot be shown. |
| Problem 2 | The category covers only 45.1% of aligned money in FY2026 and 89.1% in FY2027. |
| Problem 3 | Most of the money sits in broad groups (73.9% in one FY2027 group, 78.2% in two FY2026 groups), which do not name a type of work. |
| Options considered | Keep (FY2027 only), change, or replace with a question that works across all three plans |
| Decision | Replace (17 November 2026). The analyst decided after reviewing the evidence with Claude. |
| New question | Which asset categories receive the most aligned planned spending? Asset Category is complete for every aligned row in all three plans. |

---
# Part 4. Iteration 2: Business Understanding (revised)

Only the items that changed are shown. Everything else stays as in Part 1.

### 4.1 What changed

| Item | Iteration 1 (original) | Iteration 2 (revised) | Reason | Evidence |
|---|---|---|---|---|
| Question 3 wording | Which types of emissions-reducing work receive the most aligned funding? | Which asset categories receive the most aligned planned spending? | The original could only be answered partly | 2.3.4 |
| Question 3 columns | Greenhouse Gas (GHG) Mitigation Tracking Category, Greenhouse Gas (GHG) Mitigation, Fiscal Year Amount | Asset Category, Greenhouse Gas (GHG) Mitigation, Financial Plan, Fiscal Year Amount | Asset Category is complete for every aligned row | 2.3.6 |
| Question 3 chart | Bar chart by tracking category | Bar chart by asset category | Same chart type, new grouping | 2.3.6 |
| Question 3 purpose | Shows what drives aligned spending, such as solar or bike lanes | Shows which kinds of assets, such as buildings or vehicles, drive aligned spending | Matches what the data can show | 2.3.6 |
| Resource columns | Included the Greenhouse Gas (GHG) Mitigation Tracking Category | Tracking category removed | No question uses it any more | 2.3.4 |
| Risks | 5 risks | New risk 5: asset categories differ across plan editions (for example, "Other" only in FY2025), so compare shares, not amounts | Found in Data Understanding | 2.3.6 |
| Reference fix | Success criteria said "four questions in 3.2" | Corrected to 1.3.2 | Typo | Notebook |
| Question 4 | Asset Category or Budget Line | Unchanged for now. Data Understanding favours Asset Category; the wording is not yet narrowed. | Open decision | 2.3.5 |

### 4.2 The four questions after iteration 2

| Question | Wording | Chart | Status |
|---|---|---|---|
| Question 1 | How is the share of planned capital spending rated aligned with greenhouse gas mitigation changing across the three plan editions? | Stacked bar by plan edition | Unchanged |
| Question 2 | How does aligned spending change across fiscal years within each plan edition? | Line chart, one line per plan edition | Unchanged |
| **Question 3** | **Which asset categories receive the most aligned planned spending?** | Bar chart by asset category | **Revised** |
| Question 4 | Where is the not-aligned and pending spending concentrated? | Bar chart by asset category (recommended) or budget line | Unchanged, wording to confirm |

---
# Part 5. Iteration 2: Data Understanding (new Question 3)

| Item | Detail |
|---|---|
| Question | Which asset categories receive the most aligned planned spending? |
| Why it is valid | Asset Category is complete for every aligned row and is used in all three plans. It pairs with Question 4, which shows where the not-aligned and pending money sits. |
| Chart | A bar chart of aligned spending by asset category. |

#### 2.3.6 Question 3 check (replacement): Asset categories receiving the most aligned planned spending

| Item | Detail |
|---|---|
| Columns used | `asset_category` (merged into `asset_category_clean`), `ghg_mitigation`, `financial_plan`, `fiscal_year_amount` (thousands of US dollars) |
| Assumption | Aligned means both Aligned and Aligned Component. |
| Evidence | The aligned amount by asset category and plan, each category's share of the plan's aligned money, and the aligned share of each category's own total. |
| Consistency check | Column totals match the Question 1 aligned totals for each plan. No aligned row has a blank asset category. |

**Observations**

| Topic | Observation |
|---|---|
| Concentration | Buildings & Facilities holds 59.6%, 61.1% and 73.8% of each plan's aligned money (7.2, 13.8 and 21.9 billion dollars). |
| Change in the FY2027 plan | Compared with the FY2026 plan, aligned money rises in Buildings & Facilities (by about 8.0 billion dollars) and Special Projects (by about 1.3 billion) and falls in most other categories. Plans cover different years, so compare shares across plans, not amounts. |
| Share of own total | Energy & Utilities is 38.9%, 70.1% and 87.8% aligned. In FY2027, Buildings & Facilities is 46.2%, Green & Natural Space 35.2% and Equipment 31.5%. IT & Technology (about 1%) and Sewers & Grey Infrastructure (about 0.2%) are close to zero. |
| Category lists differ by plan | "Other" holds 22.5% of aligned money in FY2025 and does not appear later. "Special Projects" appears as an asset category in FY2026 and FY2027 only. |
| Labels | "It & Technology" was merged into "IT & Technology" (same category, different capital letters). |

**Decisions**

| Topic | Decision |
|---|---|
| Question 3 replaced | The original question (tracking categories) is replaced by this one. See 2.3.4 for the reasons. |
| Chart view | To be decided. Recommended: the FY2027 plan, aligned money by asset category in billions of US dollars, in the same category order as the Question 4 chart. |

**Can the data answer Question 3?**

| Answer | Detail |
|---|---|
| Yes. | For all three plans. Comparisons across plans should use shares, and categories that exist in only some plans need a note. |

---
# Part 6. Where things stand

### 6.1 Status of each question

| Question | Business Understanding | Data Understanding | Open decision |
|---|---|---|---|
| Question 1 | Agreed | Done: answerable with caveats | Chart is a stacked bar by plan edition |
| Question 2 | Agreed (reworded to aligned spending) | Done: answerable | Show one line per plan, shares, or 2026 to 2029 only |
| Question 3 | Revised in iteration 2 | Done: answerable for all three plans | Show the FY2027 plan in billions (recommended), or shares |
| Question 4 | Agreed | Done: answerable with asset category | Narrow the wording to Asset Category; show FY2027 shares (recommended) |

### 6.2 Next steps

| Step | What | Why |
|---|---|---|
| 2.4 | Verify data quality (missing values, inconsistent labels, duplicates, flag casing) | Confirms the data is reliable before preparation |
| 2.5 | Close the loop: write down the definition of aligned and feed the findings back to Business Understanding | Completes Data Understanding |
| Data Preparation | Clean labels, merge variants, design the database (ERM, schema) | Prepares the data for the four charts |
| Reminders | Confirm keeping the 2,108 negative amounts; check the OMB 97.7 billion dollar figure before quoting it | Accuracy of the report |

---
# Part 7. Decision log

| # | Date | Phase | Iteration | Decision |
|---|---|---|---|---|
| 1 | 10 Nov 2026 | Business Understanding | 1 | One objective: reducing emissions in line with net-zero by 2050 |
| 2 | 10 Nov 2026 | Business Understanding | 1 | One analytics criterion and one business criterion kept |
| 3 | 10 Nov 2026 | Business Understanding | 1 | Reproduce each chart by querying the database and plotting with Matplotlib |
| 4 | 10 Nov 2026 | Business Understanding | 1 | Resources, constraints and risks structure kept; Matplotlib, Pandas and Claude named |
| 5 | 10 Nov 2026 | Business Understanding | 1 | Three constraints kept |
| 6 | 10 Nov 2026 | Business Understanding | 1 | Five risks kept |
| 7 | 10 Nov 2026 | Business Understanding | 1 | Four questions kept |
| 8 | 10 Nov 2026 | Business Understanding | 1 | Question 2 reworded to planned aligned spending |
| 9 | 10 Nov 2026 | Business Understanding | 1 | Project plan kept |
| 10 | 10 Nov 2026 | Business Understanding | 1 | Agreed to start Data Understanding |
| 11 | 13 to 15 Nov 2026 | Data Understanding | 1 | Aligned means Aligned plus Aligned Component (stated assumption) |
| 12 | 15 Nov 2026 | Data Understanding | 1 | Question 2 chart: uncovered years blank, amounts in billions |
| 13 | 16 Nov 2026 | Data Understanding | 1 | Question 3 only partly answerable; decision postponed until Question 4 was checked |
| 14 | 17 Nov 2026 | Data Understanding | 1 | Asset Category chosen for the Question 4 chart |
| 15 | 17 Nov 2026 | Data Understanding | 1 | Question 3 replaced |
| 16 | 17 Nov 2026 | Business Understanding | 2 | Question 3 block, resource columns, risks and reference updated; iteration noted |

The full log, with reasons and evidence, is kept in the Project Log document.